In [1]:
import numpy as np
import os
import json
import pprint
import shutil
from bs4 import BeautifulSoup
import pandas as pd
import soundfile as sf



In [2]:

wsj_test_json = "/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/eval/wsj_test.json"

vb_dmd_json = "/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/eval/vb_dmd.json"

In [3]:
with open(wsj_test_json) as json_file:
   wsjqut_test = json.load(json_file)

wsjqut_test_keys= list(wsjqut_test.keys())
wsjqut_test_keys.sort()
wsjqut_test={key:wsjqut_test[key] for key in wsjqut_test_keys}

with open(vb_dmd_json) as json_file:
   vbdmd_test = json.load(json_file)

vbdmd_test_keys= list(vbdmd_test.keys())
vbdmd_test_keys.sort()   
vbdmd_test={key:vbdmd_test[key] for key in vbdmd_test_keys}


In [4]:
list_noise_wsjqut = [wsjqut_test[key]["noise_type"] for key in wsjqut_test.keys()]
uniq_list_noise_wsjqut = list(set(list_noise_wsjqut))
uniq_list_noise_wsjqut.sort()
print(uniq_list_noise_wsjqut)


['CAFE-CAFE-2', 'CAR-WINUPB-2', 'HOME-LIVINGB-2', 'STREET-KG-2']


In [5]:
list_snr_wsjqut = [wsjqut_test[key]["snr"] for key in wsjqut_test.keys()]
uniq_list_snr_wsjqut = list(set(list_snr_wsjqut))
uniq_list_snr_wsjqut.sort()
print(uniq_list_snr_wsjqut)

[-5, 0, 5]


In [6]:
list_noise_vbdmd = [vbdmd_test[key]["noise_type"] for key in vbdmd_test.keys()]
uniq_list_noise_vbdmd = list(set(list_noise_vbdmd))
uniq_list_noise_vbdmd.sort()
print(uniq_list_noise_vbdmd)

['bus', 'cafe', 'living', 'office', 'psquare']


In [7]:
list_snr_vbdmd = [vbdmd_test[key]["snr"] for key in vbdmd_test.keys()]
uniq_list_snr_vbdmd = list(set(list_snr_vbdmd))
uniq_list_snr_vbdmd.sort()
print(uniq_list_snr_vbdmd)

[2.5, 7.5, 12.5, 17.5]


In [8]:
# list_noise_snr_wsjqut = []

# for i in uniq_list_noise_wsjqut:
#     for j in uniq_list_snr_wsjqut:
#         list_noise_snr_wsjqut.append((i,j))


# list_noise_snr_vbdmd = []

# for k in uniq_list_noise_vbdmd:
#     for l in uniq_list_snr_vbdmd:
#         list_noise_snr_vbdmd.append((k,l))

# # print(list_noise_snr_wsjqut)   
# # print()
# # print(list_noise_snr_vbdmd)     

In [9]:
# wsjqut_total_samples = 15

# wsjqut_nb_sample_with_last_high_snr = 3 #k*nb noise type

# wsjqut_remaining_nb_sample = wsjqut_total_samples - wsjqut_nb_sample_with_last_high_snr

# nb_sample_per_ntype_snr_duo = int(np.ceil(15/len(list_noise_snr_wsjqut)))


# last_snr=5 ## we might consider a list but here we'll consider only a single snr


# selected_dico = {}



# for duo in list_noise_snr_wsjqut: 

#     list_dico_matching_duo = [dico for dico in list(wsjqut_test.values()) if dico["noise_type"]== duo[0] and dico["snr"]== duo[1]]

#     selected_wsjqut_duo = list(np.random.choice(list_dico_matching_duo, size=nb_sample_per_ntype_snr_duo, replace=False,))
        
#     selected_dico[duo] = selected_wsjqut_duo



# ##we will reduce the number of sample that have high snr, as this would be less difficult to denoise

# ## for that 
    
# elem_of_selected_dico_with_high_snr = {duo:selected_dico[duo] for duo in selected_dico.keys() if duo[1]==last_snr}

# duo_other_with_lower_snr = list( set(selected_dico.keys()) - set(elem_of_selected_dico_with_high_snr.keys()) )

# elem_of_selected_dico_with_lower_snr = {duo:selected_dico[duo] for duo in duo_other_with_lower_snr}


# final_dict = elem_of_selected_dico_with_lower_snr.copy()



# new_elem_of_selected_dico_with_high_snr = {} ## in this one, we will select the dico with high snr that we want to keep


# L = len(elem_of_selected_dico_with_high_snr)


# ll = list(elem_of_selected_dico_with_high_snr.keys())


# r = wsjqut_nb_sample_with_last_high_snr/L


# ll *= (int(r)+1)


# ll = ll[:wsjqut_nb_sample_with_last_high_snr]


# uniq_ll = list(set(ll))


# for ul in uniq_ll:
#     new_elem_of_selected_dico_with_high_snr[ul] = []


# count = 0

# for e in ll :
#     new_elem_of_selected_dico_with_high_snr[e].append(elem_of_selected_dico_with_high_snr[e][0])
#     del elem_of_selected_dico_with_high_snr[e][0]

#     count += 1
    
#     if count == wsjqut_nb_sample_with_last_high_snr:
#         break


# final_dict.update(new_elem_of_selected_dico_with_high_snr)


# pprint.pp(final_dict)



In [8]:

def get_dictionaries(testset_dict, uniq_list_noise, uniq_list_snr, total_samples = 15,nb_sample_with_last_high_snr = 3, last_snr=5, imposed_nb_sample_per_duo =None, seed=42):


    ### for the last snr, we might consider a list but here we'll consider only a single snr

    list_noise_snr = []

    for i in uniq_list_noise:
        for j in uniq_list_snr:
            list_noise_snr.append((i,j))

    
    if imposed_nb_sample_per_duo is not None:
        nb_sample_per_ntype_snr_duo = int(np.ceil(total_samples/len(list_noise_snr)))
    else:
        nb_sample_per_ntype_snr_duo = imposed_nb_sample_per_duo


    selected_dico = {}

    np.random.seed(seed)

    for duo in list_noise_snr: 

        list_dico_matching_duo = [dico for dico in list(testset_dict.values()) if dico["noise_type"]== duo[0] and dico["snr"]== duo[1]]

        selected_wsjqut_duo = list(np.random.choice(list_dico_matching_duo, size=nb_sample_per_ntype_snr_duo, replace=False,))
            
        selected_dico[duo] = selected_wsjqut_duo



    ##we will reduce the number of sample that have high snr, as this would be less difficult to denoise

    ## for that we get into two another dicos the snr whom samples will be fully kept (low snr) those for which it will not be fully kept (high snr)
        
    elem_of_selected_dico_with_high_snr = {duo:selected_dico[duo] for duo in selected_dico.keys() if duo[1]==last_snr}

    duo_other_with_lower_snr = list( set(selected_dico.keys()) - set(elem_of_selected_dico_with_high_snr.keys()) )


    elem_of_selected_dico_with_lower_snr = {duo:selected_dico[duo] for duo in duo_other_with_lower_snr}


    final_dict = elem_of_selected_dico_with_lower_snr.copy()



    new_elem_of_selected_dico_with_high_snr = {} ## in this one, we will select the dico with high snr that we want to keep


    L = len(elem_of_selected_dico_with_high_snr)

    
    ll = list(elem_of_selected_dico_with_high_snr.keys())


    r = nb_sample_with_last_high_snr/L


    ll *= (int(r)+1) ## useful if we will have to come back multiple time to an element of the list


    ll = ll[:nb_sample_with_last_high_snr]


    uniq_ll = list(set(ll))


    ##initialize the dico that gather the high snr that we 'll take.
    for ul in uniq_ll:
        new_elem_of_selected_dico_with_high_snr[ul] = []


    count = 0

    for e in ll :
        new_elem_of_selected_dico_with_high_snr[e].append(elem_of_selected_dico_with_high_snr[e][0])
        del elem_of_selected_dico_with_high_snr[e][0]

        count += 1
        
        if count == nb_sample_with_last_high_snr:
            break


    final_dict.update(new_elem_of_selected_dico_with_high_snr)

    final_list  = []
    
    for key in final_dict.keys(): 
        
        final_list.extend(final_dict[key])

    #pprint.pp(final_dict)
    return final_dict, final_list



In [9]:

def simplified_get_dictionaries(testset_dict, uniq_list_noise, uniq_list_snr, imposed_nb_sample_per_duo =2,  seed=42):


    ### for the last snr, we might consider a list but here we'll consider only a single snr
    
    assert imposed_nb_sample_per_duo is not None

    list_noise_snr = []

    for i in uniq_list_noise:
        for j in uniq_list_snr:
            list_noise_snr.append((i,j))


    nb_sample_per_ntype_snr_duo = imposed_nb_sample_per_duo


    selected_dico = {}

    final_list = []
    

    for duo in list_noise_snr: 

        list_dico_matching_duo = [dico for dico in list(testset_dict.values()) if dico["noise_type"]== duo[0] and dico["snr"]== duo[1]]

        np.random.seed(seed)
        
        selected_wsjqut_duo = list(np.random.choice(list_dico_matching_duo, size=nb_sample_per_ntype_snr_duo, replace=False,))
            
        selected_dico[duo] = selected_wsjqut_duo

        final_list.extend(selected_dico[duo])

    
    return selected_dico, final_list


In [12]:
# _,selection_wsj_qut = get_dictionaries(testset_dict = wsjqut_test, uniq_list_noise=uniq_list_noise_wsjqut, uniq_list_snr=[-5,0,5], total_samples = 15,
#                    nb_sample_with_last_high_snr = 3, last_snr=5, seed=42)

# len(selection_wsj_qut)

In [13]:
# _, selection_vb_dmd = get_dictionaries(testset_dict=vbdmd_test, uniq_list_noise=uniq_list_noise_vbdmd, uniq_list_snr=[2.5, 7.5, 12.5], total_samples = 15,
#                    nb_sample_with_last_high_snr = 3, last_snr=12.5, seed=42)

# len(selection_vb_dmd)

In [10]:
_, selection_wsj_qut = simplified_get_dictionaries(testset_dict = wsjqut_test, uniq_list_noise=uniq_list_noise_wsjqut, uniq_list_snr=[-5,0], imposed_nb_sample_per_duo =2, seed=42)

len(selection_wsj_qut)

16

In [11]:
pprint.pp(selection_wsj_qut)

[{'p_id': '440',
  'utt_name': '440c020t',
  'noisy_wav': '{noisy_root}/440c020t_CAFE-CAFE-2_-5.wav',
  'clean_wav': '{clean_root}/440/440c020t.wav',
  'length': 4.868375,
  'noise_type': 'CAFE-CAFE-2',
  'noise_start': 24783615,
  'snr': -5},
 {'p_id': '441',
  'utt_name': '441c020h',
  'noisy_wav': '{noisy_root}/441c020h_CAFE-CAFE-2_-5.wav',
  'clean_wav': '{clean_root}/441/441c020h.wav',
  'length': 6.835125,
  'noise_type': 'CAFE-CAFE-2',
  'noise_start': 15188900,
  'snr': -5},
 {'p_id': '446',
  'utt_name': '446o030t',
  'noisy_wav': '{noisy_root}/446o030t_CAFE-CAFE-2_0.wav',
  'clean_wav': '{clean_root}/446/446o030t.wav',
  'length': 10.0385625,
  'noise_type': 'CAFE-CAFE-2',
  'noise_start': 22784014,
  'snr': 0},
 {'p_id': '447',
  'utt_name': '447c020v',
  'noisy_wav': '{noisy_root}/447c020v_CAFE-CAFE-2_0.wav',
  'clean_wav': '{clean_root}/447/447c020v.wav',
  'length': 8.3199375,
  'noise_type': 'CAFE-CAFE-2',
  'noise_start': 10910079,
  'snr': 0},
 {'p_id': '440',
  'utt_n

In [12]:
_, selection_vb_dmd = get_dictionaries(testset_dict=vbdmd_test, imposed_nb_sample_per_duo =2, uniq_list_noise=uniq_list_noise_vbdmd, uniq_list_snr=[2.5, 7.5], total_samples = 15,
                   nb_sample_with_last_high_snr = 6, last_snr=7.5, seed=42) 

len(selection_vb_dmd)

16

In [13]:
pprint.pp(selection_vb_dmd)

[{'p_id': 'p257',
  'utt_name': 'p257_058',
  'noisy_wav': '{noisy_root}/p257_058.wav',
  'clean_wav': '{clean_root}/p257_058.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 1.93},
 {'p_id': 'p232',
  'utt_name': 'p232_312',
  'noisy_wav': '{noisy_root}/p232_312.wav',
  'clean_wav': '{clean_root}/p232_312.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 2.77},
 {'p_id': 'p232',
  'utt_name': 'p232_211',
  'noisy_wav': '{noisy_root}/p232_211.wav',
  'clean_wav': '{clean_root}/p232_211.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.41},
 {'p_id': 'p232',
  'utt_name': 'p232_234',
  'noisy_wav': '{noisy_root}/p232_234.wav',
  'clean_wav': '{clean_root}/p232_234.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.6},
 {'p_id': 'p232',
  'utt_name': 'p232_072',
  'noisy_wav': '{noisy_root}/p232_072.wav',
  'clean_wav': '{clean_root}/p232_072.wav',
  'noise_type': 'cafe',
  'snr': 2.5,
  'length': 1.99},
 {'p_id': 'p257',
  'utt_name': 'p257_291',
  

In [17]:
set(C)==set(D)

TypeError: unhashable type: 'dict'

In [14]:
C = [{'p_id': 'p257',
  'utt_name': 'p257_058',
  'noisy_wav': '{noisy_root}/p257_058.wav',
  'clean_wav': '{clean_root}/p257_058.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 1.93},
 {'p_id': 'p232',
  'utt_name': 'p232_312',
  'noisy_wav': '{noisy_root}/p232_312.wav',
  'clean_wav': '{clean_root}/p232_312.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 2.77},
 {'p_id': 'p257',
  'utt_name': 'p257_106',
  'noisy_wav': '{noisy_root}/p257_106.wav',
  'clean_wav': '{clean_root}/p257_106.wav',
  'noise_type': 'bus',
  'snr': 2.5,
  'length': 2.57},
 {'p_id': 'p232',
  'utt_name': 'p232_279',
  'noisy_wav': '{noisy_root}/p232_279.wav',
  'clean_wav': '{clean_root}/p232_279.wav',
  'noise_type': 'bus',
  'snr': 2.5,
  'length': 2.32},
 {'p_id': 'p257',
  'utt_name': 'p257_175',
  'noisy_wav': '{noisy_root}/p257_175.wav',
  'clean_wav': '{clean_root}/p257_175.wav',
  'noise_type': 'living',
  'snr': 2.5,
  'length': 3.37},
 {'p_id': 'p257',
  'utt_name': 'p257_074',
  'noisy_wav': '{noisy_root}/p257_074.wav',
  'clean_wav': '{clean_root}/p257_074.wav',
  'noise_type': 'living',
  'snr': 2.5,
  'length': 1.71},
 {'p_id': 'p232',
  'utt_name': 'p232_072',
  'noisy_wav': '{noisy_root}/p232_072.wav',
  'clean_wav': '{clean_root}/p232_072.wav',
  'noise_type': 'cafe',
  'snr': 2.5,
  'length': 1.99},
 {'p_id': 'p257',
  'utt_name': 'p257_291',
  'noisy_wav': '{noisy_root}/p257_291.wav',
  'clean_wav': '{clean_root}/p257_291.wav',
  'noise_type': 'cafe',
  'snr': 2.5,
  'length': 1.7},
 {'p_id': 'p232',
  'utt_name': 'p232_211',
  'noisy_wav': '{noisy_root}/p232_211.wav',
  'clean_wav': '{clean_root}/p232_211.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.41},
 {'p_id': 'p232',
  'utt_name': 'p232_234',
  'noisy_wav': '{noisy_root}/p232_234.wav',
  'clean_wav': '{clean_root}/p232_234.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.6},
 {'p_id': 'p232',
  'utt_name': 'p232_087',
  'noisy_wav': '{noisy_root}/p232_087.wav',
  'clean_wav': '{clean_root}/p232_087.wav',
  'noise_type': 'bus',
  'snr': 7.5,
  'length': 1.98},
 {'p_id': 'p257',
  'utt_name': 'p257_085',
  'noisy_wav': '{noisy_root}/p257_085.wav',
  'clean_wav': '{clean_root}/p257_085.wav',
  'noise_type': 'bus',
  'snr': 7.5,
  'length': 2.53},
 {'p_id': 'p257',
  'utt_name': 'p257_378',
  'noisy_wav': '{noisy_root}/p257_378.wav',
  'clean_wav': '{clean_root}/p257_378.wav',
  'noise_type': 'office',
  'snr': 7.5,
  'length': 1.72},
 {'p_id': 'p257',
  'utt_name': 'p257_214',
  'noisy_wav': '{noisy_root}/p257_214.wav',
  'clean_wav': '{clean_root}/p257_214.wav',
  'noise_type': 'living',
  'snr': 7.5,
  'length': 2.61},
 {'p_id': 'p232',
  'utt_name': 'p232_154',
  'noisy_wav': '{noisy_root}/p232_154.wav',
  'clean_wav': '{clean_root}/p232_154.wav',
  'noise_type': 'cafe',
  'snr': 7.5,
  'length': 2.34},
 {'p_id': 'p232',
  'utt_name': 'p232_103',
  'noisy_wav': '{noisy_root}/p232_103.wav',
  'clean_wav': '{clean_root}/p232_103.wav',
  'noise_type': 'psquare',
  'snr': 7.5,
  'length': 3.55}]

In [15]:
D= [{'p_id': 'p257',
  'utt_name': 'p257_058',
  'noisy_wav': '{noisy_root}/p257_058.wav',
  'clean_wav': '{clean_root}/p257_058.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 1.93},
 {'p_id': 'p232',
  'utt_name': 'p232_312',
  'noisy_wav': '{noisy_root}/p232_312.wav',
  'clean_wav': '{clean_root}/p232_312.wav',
  'noise_type': 'office',
  'snr': 2.5,
  'length': 2.77},
 {'p_id': 'p232',
  'utt_name': 'p232_211',
  'noisy_wav': '{noisy_root}/p232_211.wav',
  'clean_wav': '{clean_root}/p232_211.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.41},
 {'p_id': 'p232',
  'utt_name': 'p232_234',
  'noisy_wav': '{noisy_root}/p232_234.wav',
  'clean_wav': '{clean_root}/p232_234.wav',
  'noise_type': 'psquare',
  'snr': 2.5,
  'length': 1.6},
 {'p_id': 'p232',
  'utt_name': 'p232_072',
  'noisy_wav': '{noisy_root}/p232_072.wav',
  'clean_wav': '{clean_root}/p232_072.wav',
  'noise_type': 'cafe',
  'snr': 2.5,
  'length': 1.99},
 {'p_id': 'p257',
  'utt_name': 'p257_291',
  'noisy_wav': '{noisy_root}/p257_291.wav',
  'clean_wav': '{clean_root}/p257_291.wav',
  'noise_type': 'cafe',
  'snr': 2.5,
  'length': 1.7},
 {'p_id': 'p257',
  'utt_name': 'p257_106',
  'noisy_wav': '{noisy_root}/p257_106.wav',
  'clean_wav': '{clean_root}/p257_106.wav',
  'noise_type': 'bus',
  'snr': 2.5,
  'length': 2.57},
 {'p_id': 'p232',
  'utt_name': 'p232_279',
  'noisy_wav': '{noisy_root}/p232_279.wav',
  'clean_wav': '{clean_root}/p232_279.wav',
  'noise_type': 'bus',
  'snr': 2.5,
  'length': 2.32},
 {'p_id': 'p257',
  'utt_name': 'p257_175',
  'noisy_wav': '{noisy_root}/p257_175.wav',
  'clean_wav': '{clean_root}/p257_175.wav',
  'noise_type': 'living',
  'snr': 2.5,
  'length': 3.37},
 {'p_id': 'p257',
  'utt_name': 'p257_074',
  'noisy_wav': '{noisy_root}/p257_074.wav',
  'clean_wav': '{clean_root}/p257_074.wav',
  'noise_type': 'living',
  'snr': 2.5,
  'length': 1.71},
 {'p_id': 'p232',
  'utt_name': 'p232_087',
  'noisy_wav': '{noisy_root}/p232_087.wav',
  'clean_wav': '{clean_root}/p232_087.wav',
  'noise_type': 'bus',
  'snr': 7.5,
  'length': 1.98},
 {'p_id': 'p257',
  'utt_name': 'p257_085',
  'noisy_wav': '{noisy_root}/p257_085.wav',
  'clean_wav': '{clean_root}/p257_085.wav',
  'noise_type': 'bus',
  'snr': 7.5,
  'length': 2.53},
 {'p_id': 'p257',
  'utt_name': 'p257_214',
  'noisy_wav': '{noisy_root}/p257_214.wav',
  'clean_wav': '{clean_root}/p257_214.wav',
  'noise_type': 'living',
  'snr': 7.5,
  'length': 2.61},
 {'p_id': 'p257',
  'utt_name': 'p257_378',
  'noisy_wav': '{noisy_root}/p257_378.wav',
  'clean_wav': '{clean_root}/p257_378.wav',
  'noise_type': 'office',
  'snr': 7.5,
  'length': 1.72},
 {'p_id': 'p232',
  'utt_name': 'p232_103',
  'noisy_wav': '{noisy_root}/p232_103.wav',
  'clean_wav': '{clean_root}/p232_103.wav',
  'noise_type': 'psquare',
  'snr': 7.5,
  'length': 3.55},
 {'p_id': 'p232',
  'utt_name': 'p232_154',
  'noisy_wav': '{noisy_root}/p232_154.wav',
  'clean_wav': '{clean_root}/p232_154.wav',
  'noise_type': 'cafe',
  'snr': 7.5,
  'length': 2.34}]

### creating folders that will collect the selected files

In [14]:
%%bash

mkdir /srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples

mkdir /srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/matched_vbdmd

mkdir /srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/matched_wsj0qut

mkdir /srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/mismatched_wsj0qut

mkdir /srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/mismatched_vbdmd


In [15]:

WSJ_matched_ORI = {

"SGMSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/enhanced_audio_only/audio_only_wsj0_6M_renamed/",
"Conv-TasNet":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/brever/enhancement/test_wsj_with_convt_ckpt_wsj/",
"RemixIT":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/remixit_udase_baseline/enhancement/wsjqut_with_wsjqut_ckpt_every30/",

"RVAE": "/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_wsjqut_with_wsj_ckpt/", ## "RVAE": "/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_wsjqut_with_wsj_ckpt/metrics_dnsmos_torch.csv", ideally
    

"UDiffSE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_wsj0/speech_modelling_default_6M/WSJ0/udiffse/1.5/udiffuse/",
"UDiffSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/fudiffse/test_wsjqut_with_wsj_ckpt/speech_modelling_default_6M/WSJ0/fudiffse/1.5/fudiffuse_bs4/",  #metrics_dnsmos_torch


"DEPSE-IL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_il/test_wsjqut_with_wsj_ckpt/speech_modelling_default_6M/WSJ0/depse_il/0.0/depse_il/",
"DEPSE-TL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_tl/test_wsjqut_with_wsj_ckpt/speech_modelling_default_6M/WSJ0/depse_tl/0.0/depse_tl/",


"DiffUSEEN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_wsj0/speech_modelling_default_6M/WSJ0/fudiffse_v2/1.75/fudiffse_v2_bs4/",

"ParaDiffUSE-IN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/paradiffuse_v1/check_wsjqut_with_ckpt_wsjqut/bis_200epochs_training_wsj_speech_noise_modelling_default_6M/WSJ0/paradiffuse_v1/1.0/paradiffuse_v1/",
"ParaDiffUSE-EN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/check_wsjqut_with_ckpt_wsjqut/bis_200epochs_training_wsj_speech_noise_modelling_default_6M/WSJ0/paradiffuse/5.75/paradiffuse/",

}



WSJ_mismatched_ORI  = {
"SGMSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/sgmse_supervision_loss/enhancement/sgmse_wsj_with_ckpt_vb/",
"Conv-TasNet":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/brever/enhancement/test_wsj_with_convt_ckpt_vb/",
"RemixIT":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/remixit_udase_baseline/enhancement/wsjqut_with_vbdmd_ckpt_every_30/",
"RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_wsjqut_with_vb_ckpt/",  ## "RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_wsjqut_with_vb_ckpt/metrics_dnsmos_torch.csv",
"UDiffSE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_wsj_with_ckpt_vb/my_vb_speech_modelling_default_6M/WSJ0/udiffse/1.5/udiffuse/",
"UDiffSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_wsj_with_ckpt_vb/my_vb_speech_modelling_default_6M/WSJ0/fudiffse/1.5/fudiffuse_bs4/",   

"DEPSE-IL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_il/test_wsjqut_with_vb_ckpt/my_vb_speech_modelling_default_6M/WSJ0/depse_il/0.0/depse_il/",
"DEPSE-TL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_tl/test_wsjqut_with_vb_ckpt/my_vb_speech_modelling_default_6M/WSJ0/depse_tl/0.0/depse_tl/",

"DiffUSEEN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_wsj_with_ckpt_vb/my_vb_speech_modelling_default_6M/WSJ0/fudiffse_v2/1.75/fudiffse_v2_bs4/",


"ParaDiffUSE-IN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/paradiffuse_v1/check_wsjqut_with_ckpt_vbdmd/bis_200epochs_training_vb_speech_noise_modelling_default_6M/WSJ0/paradiffuse_v1/1.0/paradiffuse_v1/",

"ParaDiffUSE-EN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/check_wsjqut_with_ckpt_vbdmd/bis_200epochs_training_vb_speech_noise_modelling_default_6M/WSJ0/paradiffuse/5.75/paradiffuse/",
}


VB_matched_ORI = {
"SGMSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/sgmse_supervision_loss/enhancement/sgmse_vb_with_ckpt_vb/",
"Conv-TasNet":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/brever/enhancement/test_vb_with_convt_ckpt_vb/",
"RemixIT":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/remixit_udase_baseline/enhancement/vbdmd_with_vbdmd_ckpt_every30/",
"RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_vbdmd_with_vb_ckpt/",   ##"RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_vbdmd_with_vb_ckpt/metrics_dnsmos_torch.csv", ideally
"UDiffSE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb_with_ckpt_vb/my_vb_speech_modelling_default_6M/VB/udiffse/1.5/udiffuse/",
                           
"UDiffSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb_with_ckpt_vb/my_vb_speech_modelling_default_6M/VB/fudiffse/1.5/fudiffuse_bs4/",
"DEPSE-IL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_il/test_vbdmd_with_vb_ckpt/my_vb_speech_modelling_default_6M/VB/depse_il/0.0/depse_il/",
"DEPSE-TL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_tl/test_vbdmd_with_vb_ckpt/my_vb_speech_modelling_default_6M/VB/depse_tl/0.0/depse_tl/",

"DiffUSEEN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb_with_ckpt_vb/my_vb_speech_modelling_default_6M/VB/fudiffse_v2/1.75/fudiffse_v2_bs4/",
                  
"ParaDiffUSE-IN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/paradiffuse_v1/check_vbdmd_with_ckpt_vbdmd/bis_200epochs_training_vb_speech_noise_modelling_default_6M/VB/paradiffuse_v1/1.0/paradiffuse_v1/",
"ParaDiffUSE-EN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/check_vbdmd_with_ckpt_vbdmd/bis_200epochs_training_vb_speech_noise_modelling_default_6M/VB/paradiffuse/5.75/paradiffuse/",
}


VB_mismatched_ORI = {
"SGMSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/sgmse_supervision_loss/enhancement/sgmse_vb_with_ckpt_wsj_qut_6M/",
"Conv-TasNet":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/brever/enhancement/test_vb_with_convt_ckpt_wsj/",
"RemixIT":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/remixit_udase_baseline/enhancement/vbdmd_with_wsjqut_ckpt_every30/",
"RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_vbdmd_with_wsj_ckpt/", ###"RVAE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/DVAE_SE/enhancement/test_vbdmd_with_wsj_ckpt/metrics_dnsmos_torch.csv",
"UDiffSE":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb/speech_modelling_default_6M/VB/udiffse/1.5/udiffuse/",
"UDiffSE+":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb/speech_modelling_default_6M/VB/fudiffse/1.5/fudiffuse_bs4/",

"DEPSE-IL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_il/test_vbdmd_with_wsj_ckpt/speech_modelling_default_6M/VB/depse_il/0.0/depse_il/",
"DEPSE-TL":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/ieee_letter/depse_tl/test_vbdmd_with_wsj_ckpt/speech_modelling_default_6M/VB/depse_tl/0.0/depse_tl/",
"DiffUSEEN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/diffuse_noise/eval/EXTENDED/check_vb/speech_modelling_default_6M/VB/fudiffse_v2/1.75/fudiffse_v2_bs4/",

"ParaDiffUSE-IN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/paradiffuse_v1/check_vbdmd_with_ckpt_wsjqut/bis_200epochs_training_wsj_speech_noise_modelling_default_6M/VB/paradiffuse_v1/1.0/paradiffuse_v1/",
"ParaDiffUSE-EN":"/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/conditional_para/eval/EXTENDED/bis_training_epoch_200/check_vbdmd_with_ckpt_wsjqut/bis_200epochs_training_wsj_speech_noise_modelling_default_6M/VB/paradiffuse/5.75/paradiffuse/",
}



In [16]:
def get_files_into_folder(list_selected_demo, path_dico_storage, saving_directory, dataset):


    if dataset == "VB":
        noisy_dir = "/srv/storage/talc@storage4.nancy.grid5000.fr/multispeech/corpus/source_separation/VoiceBankDEMAND/noisy_testset_wav_16k"
        clean_dir = "/srv/storage/talc@storage4.nancy.grid5000.fr/multispeech/corpus/source_separation/VoiceBankDEMAND/clean_testset_wav_16k"
    
    elif dataset == "WSJ0":
        noisy_dir =  "/srv/storage/talc@storage4.nancy.grid5000.fr/multispeech/corpus/source_separation/QUT_WSJ0/test"
        clean_dir =  "/srv/storage/talc@storage4.nancy.grid5000.fr/multispeech/corpus/source_separation/WSJ0_SE/wsj0_si_et_05"        


    for dico in list_selected_demo:

        subfolder = f"{dico['utt_name']}_{dico['noise_type']}_{dico['snr']}"

        os.makedirs(os.path.join(saving_directory, subfolder ))

        if dataset == "WSJ0":
            g, sr = sf.read(dico['clean_wav'].format(clean_root=clean_dir))

            assert sr==16000

            sf.write(os.path.join(saving_directory,subfolder, f"clean.wav"), g, 16000)            
        
        else:
            shutil.copy(dico['clean_wav'].format(clean_root=clean_dir), os.path.join(saving_directory,subfolder, f"clean.wav") )
                 
        shutil.copy(dico['noisy_wav'].format(noisy_root=noisy_dir), os.path.join(saving_directory,subfolder, f"noisy.wav") )   

        
        for key,value in path_dico_storage.items():

            shutil.copy(os.path.join(value,f'{dico["utt_name"]}.wav') , os.path.join(saving_directory,subfolder,f"{key}.wav") )   ## rename like udiffse_id_ntype_snr

    #_{dico['utt_name']}_{dico['noise_type']}_{dico['snr']}




In [17]:
get_files_into_folder(list_selected_demo=selection_wsj_qut,
                      path_dico_storage=WSJ_matched_ORI, 
                      saving_directory="/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/matched_wsj0qut", 
                      dataset="WSJ0")

In [18]:
get_files_into_folder(list_selected_demo=selection_wsj_qut,
                      path_dico_storage=WSJ_mismatched_ORI, 
                      saving_directory="/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/mismatched_wsj0qut", 
                      dataset="WSJ0")

In [20]:
get_files_into_folder(list_selected_demo=selection_vb_dmd,
                      path_dico_storage=VB_matched_ORI, 
                      saving_directory="/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/matched_vbdmd", 
                      dataset="VB")

In [19]:
get_files_into_folder(list_selected_demo=selection_vb_dmd,
                      path_dico_storage=VB_mismatched_ORI, 
                      saving_directory="/srv/storage/talc3@storage4.nancy.grid5000.fr/multispeech/calcul/users/jayilo/proper_diffuse_noise/demo_samples/mismatched_vbdmd", 
                      dataset="VB")

In [21]:
def get_htlm_dico(dataset,media_type):    

    dico={}

    list_media = [f"clean.{media_type}", f"noisy.{media_type}", f"SGMSE+.{media_type}", f"Conv-TasNet.{media_type}", f"RemixIT.{media_type}", f"RVAE.{media_type}", f"UDiffSE.{media_type}", f"UDiffSE+.{media_type}", f"DEPSE-IL.{media_type}", f"DEPSE-TL.{media_type}", f"DiffUSEEN.{media_type}", f"ParaDiffUSE-IN.{media_type}", f"ParaDiffUSE-EN.{media_type}"]

    samples = os.listdir(f"./demo_samples/{dataset}")

    ##collect in a dico the name of the sample and its corresponding clean, noisy and enhanced files
    for folder in samples :

        name_elements = folder.split('_') #e.g. :'09F_SPSQUARE_-5_sx374' for example
        
        ## in windows based os , os.path.join lead to the use of "\" instead of "/" in the path, so we preferred manuallly write it
        #dico[f"{name_elements[1]} {name_elements[2]}"] = [os.path.join(f"./icp_samples/{dataset}",media ) for media in list_media]
        
        #we comment this line because with just {name_elements[1]} ({name_elements[2]}), the directionary keys are not unique, and this reduce the number of entries in the dict
        #dico[f"{name_elements[1]} ({name_elements[2]})"] = [f"./icp_samples/{dataset}" + "/"+ media  for media in list_media]
        
        ##add a special character to the path so that we can easily identify in the html table, cell that contains file path
        special_character = "!-!"
        dico[folder] = [f"{special_character} ./demo_samples/{dataset}/{folder}" + "/"+ media  for media in list_media]
        

    df = pd.DataFrame.from_dict(data=dico, orient='index', columns = ["Clean", "Noisy", "SGMSE+ [1]", "Conv-TasNet [2]", "RemixIT [3]","RVAE [4]", "UDiffSE [5]", "UDiffSE+ [6]", "DEPSE-IL [7]", "DEPSE-TL [7]", "DiffUSEEN", "ParaDiffUSE-IN", "ParaDiffUSE-EN"])
    df.reset_index(inplace=True)          
    df.rename(columns={"index":"id_noise_snr"},inplace=True)

    df_html_code = df.to_html(escape=False, index=False)
    
    df_html_code = BeautifulSoup(df_html_code)
    
    # print(df_html_code)

    ## replace the <td> special_character path </td> with <td><video width="150" height="150" controls><source src={cell.find(text=True).replace(f"{special_character} ", "")} type="video/mp4"></video><td>
    
    for cell in df_html_code.find_all("td"): #findAll

        if special_character in cell.find(string=True):        
            
            if media_type == "wav" :
                
                new_cell = BeautifulSoup(f'<td> <button class="audio-btn play" data-audio={cell.find(text=True).replace(f"{special_character} ", "")}>  </button> </td>')               
                
                # new_cell = BeautifulSoup(f'<td style="width: 16.6667%;"><audio style="width: 150px; height: 40px;" controls="controls"><source src={cell.find(text=True).replace(f"{special_character} ", "")} /></audio></td>' )               

            cell.replace_with(new_cell)    
    

    print(df_html_code)  

    return df_html_code, df

    

In [22]:
_, _ =get_htlm_dico(dataset="matched_wsj0qut",
              media_type="wav")

<table border="1" class="dataframe">
<thead>
<tr style="text-align: right;">
<th>id_noise_snr</th>
<th>Clean</th>
<th>Noisy</th>
<th>SGMSE+ [1]</th>
<th>Conv-TasNet [2]</th>
<th>RemixIT [3]</th>
<th>RVAE [4]</th>
<th>UDiffSE [5]</th>
<th>UDiffSE+ [6]</th>
<th>DEPSE-IL [7]</th>
<th>DEPSE-TL [7]</th>
<th>DiffUSEEN</th>
<th>ParaDiffUSE-IN</th>
<th>ParaDiffUSE-EN</th>
</tr>
</thead>
<tbody>
<tr>
<td>441c020h_CAFE-CAFE-2_-5</td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_wsj0qut/441c020h_CAFE-CAFE-2_-5/clean.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_wsj0qut/441c020h_CAFE-CAFE-2_-5/noisy.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_wsj0qut/441c020h_CAFE-CAFE-2_-5/SGMSE+.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_wsj0qut/441c020h_CAFE-CAFE-2_-5/Conv-TasNet.wav"> </button> </td>
<td> <button class="audio-btn play" data-

/tmp/ipykernel_2810275/1314490802.py:43: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  new_cell = BeautifulSoup(f'<td> <button class="audio-btn play" data-audio={cell.find(text=True).replace(f"{special_character} ", "")}>  </button> </td>')


In [23]:
_,_ = get_htlm_dico(dataset="mismatched_wsj0qut",
              media_type="wav")

<table border="1" class="dataframe">
<thead>
<tr style="text-align: right;">
<th>id_noise_snr</th>
<th>Clean</th>
<th>Noisy</th>
<th>SGMSE+ [1]</th>
<th>Conv-TasNet [2]</th>
<th>RemixIT [3]</th>
<th>RVAE [4]</th>
<th>UDiffSE [5]</th>
<th>UDiffSE+ [6]</th>
<th>DEPSE-IL [7]</th>
<th>DEPSE-TL [7]</th>
<th>DiffUSEEN</th>
<th>ParaDiffUSE-IN</th>
<th>ParaDiffUSE-EN</th>
</tr>
</thead>
<tbody>
<tr>
<td>441c020h_CAFE-CAFE-2_-5</td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_wsj0qut/441c020h_CAFE-CAFE-2_-5/clean.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_wsj0qut/441c020h_CAFE-CAFE-2_-5/noisy.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_wsj0qut/441c020h_CAFE-CAFE-2_-5/SGMSE+.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_wsj0qut/441c020h_CAFE-CAFE-2_-5/Conv-TasNet.wav"> </button> </td>
<td> <button class="audio-btn

/tmp/ipykernel_2810275/1314490802.py:43: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  new_cell = BeautifulSoup(f'<td> <button class="audio-btn play" data-audio={cell.find(text=True).replace(f"{special_character} ", "")}>  </button> </td>')


In [24]:
_,_ = get_htlm_dico(dataset=f"matched_vbdmd",
              media_type="wav")

<table border="1" class="dataframe">
<thead>
<tr style="text-align: right;">
<th>id_noise_snr</th>
<th>Clean</th>
<th>Noisy</th>
<th>SGMSE+ [1]</th>
<th>Conv-TasNet [2]</th>
<th>RemixIT [3]</th>
<th>RVAE [4]</th>
<th>UDiffSE [5]</th>
<th>UDiffSE+ [6]</th>
<th>DEPSE-IL [7]</th>
<th>DEPSE-TL [7]</th>
<th>DiffUSEEN</th>
<th>ParaDiffUSE-IN</th>
<th>ParaDiffUSE-EN</th>
</tr>
</thead>
<tbody>
<tr>
<td>p257_267_bus_2.5</td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_vbdmd/p257_267_bus_2.5/clean.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_vbdmd/p257_267_bus_2.5/noisy.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_vbdmd/p257_267_bus_2.5/SGMSE+.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_vbdmd/p257_267_bus_2.5/Conv-TasNet.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/matched_vbdmd/p257_26

/tmp/ipykernel_2810275/1314490802.py:43: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  new_cell = BeautifulSoup(f'<td> <button class="audio-btn play" data-audio={cell.find(text=True).replace(f"{special_character} ", "")}>  </button> </td>')


In [25]:
_,_ = get_htlm_dico(dataset="mismatched_vbdmd",
              media_type="wav")

<table border="1" class="dataframe">
<thead>
<tr style="text-align: right;">
<th>id_noise_snr</th>
<th>Clean</th>
<th>Noisy</th>
<th>SGMSE+ [1]</th>
<th>Conv-TasNet [2]</th>
<th>RemixIT [3]</th>
<th>RVAE [4]</th>
<th>UDiffSE [5]</th>
<th>UDiffSE+ [6]</th>
<th>DEPSE-IL [7]</th>
<th>DEPSE-TL [7]</th>
<th>DiffUSEEN</th>
<th>ParaDiffUSE-IN</th>
<th>ParaDiffUSE-EN</th>
</tr>
</thead>
<tbody>
<tr>
<td>p257_267_bus_2.5</td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_vbdmd/p257_267_bus_2.5/clean.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_vbdmd/p257_267_bus_2.5/noisy.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_vbdmd/p257_267_bus_2.5/SGMSE+.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatched_vbdmd/p257_267_bus_2.5/Conv-TasNet.wav"> </button> </td>
<td> <button class="audio-btn play" data-audio="./demo_samples/mismatche

/tmp/ipykernel_2810275/1314490802.py:43: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  new_cell = BeautifulSoup(f'<td> <button class="audio-btn play" data-audio={cell.find(text=True).replace(f"{special_character} ", "")}>  </button> </td>')
